# DA5301W: Python for Data Science
## Assignment 2 (Take-Home): Social Network Graph Search

**Department of Data Science and AI (DSAI), IIT Madras**  
**Topic:** Graph Search on SNAP Facebook Friendship Network  
**Total Marks:** 100 Marks  

---
### Overview & Guidelines
- **Dataset**: Stanford SNAP Facebook Friendship Network (`facebook_combined.txt`).
- **Exercise 1 [40 Marks]**: Uninformed Search — Breadth-First Search (BFS) and Depth-First Search (DFS).
  - *Note: You may directly reuse or adapt your BFS and DFS implementations from the In-Class assignment.*
- **Exercise 2 [60 Marks]**: Informed Search — A* Search on Weighted Social Network with Jaccard Friction Weights.
- **Evaluation**: Evaluated strictly on the 2 benchmark test cases per algorithm (No private test cases, no external plotting required).

## Exercise 0: Setup and Dataset Ingestion with NumPy

Load `data/facebook_combined.txt` using `np.loadtxt` and verify network statistics.

In [1]:
import os
import heapq
from collections import deque
import numpy as np
import networkx as nx

#  Check dataset path
data_path=os.path.join("..","data","facebook_combined.txt")
if not os.path.exists(data_path):
    raise FileNotFoundError(f"Dataset not found at {data_path}. Please place facebook_combined.txt in data/")

edges=np.loadtxt(data_path,dtype=int)

print(f"Loaded {len(edges)} edges using np.loadtxt.")
print(f"Node ID range: min = {edges.min()}, max = {edges.max()}")

# adjacency dict, neighbours sorted
adj={}
for a,b in edges.tolist():
    adj.setdefault(a,set()).add(b)
    adj.setdefault(b,set()).add(a)
adj={u:sorted(v) for u,v in sorted(adj.items())}
print("Nodes:",len(adj),"| Edges:",sum(len(v) for v in adj.values())//2)

# Verify with NetworkX
G=nx.Graph()
G.add_edges_from(edges.tolist())
print("NetworkX check:",G.number_of_nodes(),"nodes,",G.number_of_edges(),"edges")
assert len(adj)==4039 and G.number_of_edges()==88234

Loaded 88234 edges using np.loadtxt.
Node ID range: min = 0, max = 4038
Nodes: 4039 | Edges: 88234


NetworkX check: 4039 nodes, 88234 edges


## Exercise 1: Uninformed Search — BFS and DFS [40 Marks]

*Note: You may directly reuse and adapt your `bfs` and `dfs` implementations from the In-Class assignment.*

1. **BFS Evaluation** [20 Marks]: `bfs(graph, start, goal)` returning `(path, expansions)` [10 Marks per test case].
2. **DFS Evaluation** [20 Marks]: `dfs(graph, start, goal)` returning `(path, expansions)` [10 Marks per test case].

**Test Cases**:
- **Test Case 1**: `(107, 1684)` (Direct shortcut between top two hubs)
- **Test Case 2**: `(107, 692)` (Referral route to the furthest node in the network)

*Note: Inspect neighbors in ascending order of node ID for deterministic expansion order.*

In [2]:
def trace(par,t):
    p=[t]
    while par[p[-1]] is not None:
        p.append(par[p[-1]])
    return p[::-1]

def bfs(graph,start,goal):
    par={start:None}
    q=deque([start])
    exp=0
    while q:
        u=q.popleft()
        exp+=1
        if u==goal:
            return trace(par,goal),exp
        for v in sorted(graph[u]):
            if v not in par:
                par[v]=u
                q.append(v)
    return None,exp

def dfs(graph,start,goal):
    par={}
    seen=set()
    st=[(start,None)]
    exp=0
    while st:
        u,p=st.pop()
        if u in seen:
            continue
        seen.add(u)
        par[u]=p
        exp+=1
        if u==goal:
            return trace(par,goal),exp
        for v in sorted(graph[u],reverse=True):
            if v not in seen:
                st.append((v,u))
    return None,exp

# Run and benchmark on 2 Test Cases
test_pairs=[("pair_1",107,1684),("pair_2",107,692)]
res=[]
for name,f in [("BFS",bfs),("DFS",dfs)]:
    for label,s,t in test_pairs:
        p,e=f(adj,s,t)
        res.append((name,label,s,t,p,e))
        print(f"{name} {label} ({s} -> {t}):")
        print("  path:",p)
        print("  path length:",len(p)-1,"hops | expansions:",e)

BFS pair_1 (107 -> 1684):
  path: [107, 1684]
  path length: 1 hops | expansions: 819
BFS pair_2 (107 -> 692):
  path: [107, 1085, 862, 698, 686, 692]
  path length: 5 hops | expansions: 3903
DFS pair_1 (107 -> 1684):
  path: [107, 0, 1, 48, 30, 9, 3, 25, 21, 13, 26, 40, 29, 16, 82, 56, 55, 67, 10, 142, 31, 7, 22, 87, 5, 122, 45, 104, 98, 59, 118, 65, 203, 50, 109, 119, 27, 54, 53, 24, 57, 80, 88, 73, 126, 260, 170, 62, 96, 133, 141, 38, 117, 77, 231, 106, 169, 103, 136, 120, 277, 39, 69, 105, 148, 113, 66, 134, 132, 72, 165, 121, 206, 178, 186, 123, 200, 75, 85, 156, 188, 79, 185, 271, 63, 261, 172, 128, 150, 64, 100, 163, 173, 34, 348, 198, 166, 202, 329, 130, 204, 92, 94, 101, 180, 187, 194, 196, 249, 213, 161, 199, 211, 222, 238, 239, 60, 158, 168, 129, 291, 208, 285, 146, 313, 84, 51, 23, 190, 229, 334, 252, 212, 232, 224, 236, 248, 265, 223, 246, 304, 257, 258, 272, 108, 36, 127, 135, 184, 139, 159, 197, 251, 281, 284, 309, 320, 300, 299, 242, 254, 266, 302, 315, 280, 290, 322, 2

## Exercise 2: Informed Search — A* on the Weighted Social Network [60 Marks]

To prepare the social graph for A* search, compute edge weights using neighborhood Jaccard similarity.

### Pre-Search Step: Jaccard Similarity & Friction Weights [20 Marks]
1. Compute the neighborhood Jaccard similarity for every edge:
   $$J(u, v) = \frac{|N(u) \cap N(v)|}{|N(u) \cup N(v)|}$$
2. Assign integer friction weights:
   $$w(u, v) = 1 + \lfloor 10 \times (1 - J(u, v)) \rfloor \in [1, 11]$$
3. Print summary statistics of friction weights (min, max, mean).

> **Concrete Example**:
> If $N(u) = \{1, 2, 3, 4\}$ and $N(v) = \{3, 4, 5, 6, 7\}$:
> - Intersection $|N(u) \cap N(v)| = |\{3, 4\}| = 2$
> - Union $|N(u) \cup N(v)| = |\{1, 2, 3, 4, 5, 6, 7\}| = 7$
> - Jaccard $J(u, v) = 2/7 \approx 0.2857$
> - Friction weight: $w(u, v) = 1 + \lfloor 10 \times (1 - 0.2857) \rfloor = 1 + 7 = 8$.

### A* Search Implementation & 2 Benchmark Tests [40 Marks]
1. Define modular helper subroutines `extract_min(open_set)` and `reconstruct_path(parent, goal)` [10 Marks].
2. Define the **Admissible Neighborhood Heuristic** $h(u, t)$:
   $$h(u, t) = \begin{cases} 0 & \text{if } u = t \\ w(u, t) & \text{if } (u, t) \in E \\ 2 & \text{if } (u, t) \notin E \end{cases}$$
3. Implement `astar_search(graph, weights, start, goal, heuristic_fn)`.
4. Evaluate on Test Case 1 `(107, 1684)` [15 Marks] and Test Case 2 `(107, 692)` [15 Marks].

In [3]:
# degrees with numpy
deg=np.bincount(edges.ravel(),minlength=4039)
top=np.argsort(-deg,kind="stable")[:3]
print("Top 3 hubs:",[(int(n),int(deg[n])) for n in top])
print("Mean degree:",round(deg.mean(),4))

# jaccard + friction weights (integer maths, so floor has no float error)
nb={u:set(v) for u,v in adj.items()}
weights={}
for u,v in edges.tolist():
    i=len(nb[u]&nb[v])
    un=len(nb[u]|nb[v])
    w=1+10*(un-i)//un
    weights[(u,v)]=w
    weights[(v,u)]=w
ws=np.array([weights[(u,v)] for u,v in edges.tolist()])
print("Edges weighted:",len(ws))
print("Friction weights: min =",ws.min(),"| max =",ws.max(),"| mean =",round(ws.mean(),4))

Top 3 hubs: [(107, 1045), (1684, 792), (1912, 755)]
Mean degree: 43.691


Edges weighted: 88234
Friction weights: min = 1 | max = 11 | mean = 6.8193


In [4]:
def extract_min(open_set):
    return heapq.heappop(open_set)

def reconstruct_path(parent,goal):
    p=[goal]
    while parent[p[-1]] is not None:
        p.append(parent[p[-1]])
    return p[::-1]

def heuristic(u,goal,weights=None):
    if u==goal:
        return 0
    if weights is not None and (u,goal) in weights:
        return weights[(u,goal)]
    return 2

def astar_search(graph,weights,start,goal,heuristic_fn=None):
    if heuristic_fn is None:
        heuristic_fn=heuristic
    g={start:0}
    parent={start:None}
    op=[(heuristic_fn(start,goal,weights),0,start)]
    closed=set()
    exp=0
    while op:
        f,cg,u=extract_min(op)
        if u in closed:
            continue
        closed.add(u)
        exp+=1
        if u==goal:
            return reconstruct_path(parent,goal),g[goal],exp
        for v in sorted(graph[u]):
            gn=g[u]+weights[(u,v)]
            if gn<g.get(v,float("inf")):
                g[v]=gn
                parent[v]=u
                heapq.heappush(op,(gn+heuristic_fn(v,goal,weights),gn,v))
    return None,float("inf"),exp

# Benchmark A* on the 2 Test Cases
for label,s,t in test_pairs:
    a_path,a_cost,a_exp=astar_search(adj,weights,s,t)
    res.append(("A*",label,s,t,a_path,a_exp,a_cost))
    print(f"{label} ({s} -> {t}):")
    print("  path:",a_path)
    print(f"  A*: optimal cost = {a_cost}, path length = {len(a_path)-1} hops, expansions = {a_exp}")

pair_1 (107 -> 1684):
  path: [107, 1684]
  A*: optimal cost = 10, path length = 1 hops, expansions = 10
pair_2 (107 -> 692):
  path: [107, 1085, 862, 698, 686, 692]
  A*: optimal cost = 49, path length = 5 hops, expansions = 3986


## Summary of Benchmark Results

Performance comparison across all algorithms on the 2 assigned test cases.

In [5]:
print("="*70)
print("BENCHMARK SUMMARY (2 Tests Per Algorithm)")
print("="*70)
print("Algorithm | Test Case              | Length (Hops) | Cost | Expansions")
print("-"*70)
for r in res:
    name,label,s,t,p,e=r[:6]
    c=r[6] if len(r)>6 else sum(weights[(a,b)] for a,b in zip(p,p[1:]))
    print(f"{name:9} | {label} ({s}->{t}) | {len(p)-1:13} | {c:4} | {e}")

BENCHMARK SUMMARY (2 Tests Per Algorithm)
Algorithm | Test Case              | Length (Hops) | Cost | Expansions
----------------------------------------------------------------------
BFS       | pair_1 (107->1684) |             1 |   10 | 819
BFS       | pair_2 (107->692) |             5 |   49 | 3903
DFS       | pair_1 (107->1684) |           174 | 1410 | 283
DFS       | pair_2 (107->692) |           178 | 1452 | 467
A*        | pair_1 (107->1684) |             1 |   10 | 10
A*        | pair_2 (107->692) |             5 |   49 | 3986
